# C3PA QSBC — Standalone CoT Batch Runner (Colab)

Runs the QSBC Phase-1 reasoning generation against a **local** model
(Gemma 4 via Ollama) directly in Colab — no FastAPI/Postgres/opencode
needed. Parsing and prompt construction reuse the repo's exact logic, so
payloads match the API-driven batches byte-for-byte.

**Resume-safe:** sampling uses a deterministic per-label seeded shuffle
(`SEED + label_id`) and takes the next `PER_LABEL` sentences not already in
`results/`. Each run advances through the same permutation, so committing
the shard files back to GitHub lets the next session continue exactly where
this one left off. Results are written incrementally (a session death loses
at most a few rows).

**Output:** `results/reasonings_<start>-<end>.json`, one file per 500 rows
(`SHARD_SIZE`). Download them (cell 7) and commit to the repo to resume later.

**Knobs:** `MODEL`, `PER_LABEL`, `DOC_LIMIT`, `MIN_N`/`MAX_N`, `SEED`,
`RUN_ID`, `MAX_WORKERS`, `SHARD_SIZE`.


In [29]:
# 1. Clone repo + dataset
import os, sys
os.chdir('/content')
if not os.path.exists('/content/qsbc'):
    !git clone -q https://github.com/jpeckenpaugh/qsbc.git qsbc
    !git clone -q https://github.com/MaazBinMusa/C3PA_Dataset.git qsbc/C3PA_Dataset
os.chdir('/content/qsbc')
sys.path.insert(0, '/content/qsbc')
print('cwd:', os.getcwd())
print('repo files:', sorted(os.listdir('.'))[:12])
print('existing results shards:', sorted(os.listdir('results')) if os.path.isdir('results') else 'none')


cwd: /content/qsbc
repo files: ['.dockerignore', '.git', '.gitignore', '.opencode', 'C3PA_Dataset', 'C3PA_Explorer.png', 'C3PA_LR_vs_BERT_vs_FLAN_T5.ipynb', 'Dockerfile', 'Quantized_Semantic_Bottleneck_Classifier_(QSBC)_for_C3PA_Privacy_Policies.ipynb', 'README.md', 'app', 'data']
existing results shards: ['reasonings_0000-0499.json', 'reasonings_0048-0547.json', 'reasonings_0096-0595.json', 'reasonings_0144-0643.json', 'reasonings_0192-0239.json', 'reasonings_0240-0335.json']


In [30]:
# 2. Parse C3PA -> data/*.tsv (exact same pipeline as the local DB)
!python scripts/parse_c3pa.py


documents           : 400  ([('DB', 230), ('WS', 170)])
labels              : 13
sentences (unique)  : 56,914
sentence_labels     : 81,000
multi-label counts  : {1: 40505, 2: 9397, 3: 6503, 4: 383, 5: 97, 6: 28, 7: 1}
TSV output written to data/


In [31]:
# 3. Install + start Ollama, pull model
import subprocess, time, os, requests

MODEL = 'gemma4:26b-a4b-it-q4_K_M'   # 26B-A4B MoE, 18GB -> needs L4/A100 (Colab Pro)

if not os.path.exists('/usr/local/bin/ollama'):
    # zstd for the installer; pciutils/lshw so Ollama can detect the Colab GPU
    !sudo apt-get install -y -q zstd pciutils lshw
    !curl -fsSL https://ollama.com/install.sh | sh

if not os.popen('pgrep -x ollama').read().strip():
    subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL,
                     stderr=subprocess.DEVNULL)
    for _ in range(30):
        try:
            requests.get('http://localhost:11434/api/tags', timeout=2)
            break
        except Exception:
            time.sleep(2)

!ollama pull {MODEL}
print('model ready:', requests.get('http://localhost:11434/api/tags').json())



model ready: {'models': [{'name': 'gemma4:26b-a4b-it-q4_K_M', 'model': 'gemma4:26b-a4b-it-q4_K_M', 'modified_at': '2026-09-09T19:44:25.153529611Z', 'size': 17987581215, 'digest': '5571076f3d70050487b26b341705799e0ab29b808164f90d20d4cf84f699d251', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'gemma4', 'families': ['gemma4'], 'parameter_size': '25.8B', 'quantization_level': 'Q4_K_M'}, 'capabilities': ['completion', 'tools', 'thinking']}]}


In [32]:
# 4. Load parsed TSVs -> in-memory corpus (COPY-compatible text decoding)
import csv, random
from collections import defaultdict
from app.promptlib import build_payload, render_prompt

def read_tsv(path):
    rows = []
    with open(path, newline='', encoding='utf-8') as fh:
        for row in csv.reader(fh, delimiter='\t'):
            if not row:
                continue
            # undo COPY's backslash unescaping so text matches the local DB
            rows.append([c.replace('\\\\', '\\') for c in row])
    return rows

labels   = {int(i): name for i, name in read_tsv('data/labels.tsv')}
documents = {int(i): {'subset': s, 'doc_key': k}
             for i, s, n, f, k, u in read_tsv('data/documents.tsv')}
sentences = {int(i): (int(d), t) for i, d, t in read_tsv('data/sentences.tsv')}

sentence_labels = defaultdict(set)
for sid, lid in read_tsv('data/sentence_labels.tsv'):
    sentence_labels[int(sid)].add(int(lid))

# single-label only, excluding 'Others'
single = {sid: next(iter(lids)) for sid, lids in sentence_labels.items()
          if len(lids) == 1 and labels[next(iter(lids))] != 'Others'}

# (doc_id, label_id) -> ordered [(sentence_id, text)]  (id == document order)
groups = defaultdict(list)
for sid, lid in single.items():
    did, text = sentences[sid]
    groups[(did, lid)].append((sid, text))
for k in groups:
    groups[k].sort()

print('single-label sentences:', len(single), '| groups:', len(groups))


single-label sentences: 37284 | groups: 3718


In [33]:
# 5. Sample per-label: seeded shuffle, skip already-done (resume-safe)
import json, random, os, glob

PER_LABEL = 8      # new sentences per label this run
DOC_LIMIT = 15     # centered window over same-label sentences
MIN_N, MAX_N = 2, 5
SEED = 42
RESULTS_DIR = 'results'

# done-set: every sentence_id already present in committed/previous shards
done = set()
if os.path.isdir(RESULTS_DIR):
    for f in sorted(glob.glob(os.path.join(RESULTS_DIR, 'reasonings_*.json'))):
        with open(f) as fh:
            for rec in json.load(fh):
                done.add(rec['sentence_id'])
print('already done:', len(done))

by_label = defaultdict(list)
for (did, lid), items in groups.items():
    for sid, text in items:
        by_label[lid].append((sid, did, text))

tasks = []
for lid in sorted(by_label):
    rng = random.Random(SEED + lid)          # deterministic per label
    pool_list = sorted(by_label[lid])
    rng.shuffle(pool_list)
    picked = [x for x in pool_list if x[0] not in done][:PER_LABEL]
    for sid, did, text in picked:
        ordered = groups[(did, lid)]
        target_index = next(i for i, (s, t) in enumerate(ordered) if s == sid)
        tasks.append({
            'sentence_id': sid,
            'doc_id': did,
            'label_id': lid,
            'label': labels[lid],
            'doc_key': documents[did]['doc_key'],
            'doc_total': len(ordered),
            'payload': build_payload(text, labels[lid],
                                    [t for s, t in ordered], target_index,
                                    DOC_LIMIT),
        })

print('new tasks this run:', len(tasks))
if tasks:
    print('sample payload:', json.dumps(tasks[0]['payload'], ensure_ascii=False)[:200])


already done: 288
new tasks this run: 96
sample payload: {"Sentence": "Inferences drawn from other personal information.", "Label": "Categories of Personal Information Collected", "Document": ["Physical location or movements.", "Audio, electronic, visual, t


In [34]:
# 6. Run batch: per-task retry, incremental shard save (session-safe)
import requests, time, os
from concurrent.futures import ThreadPoolExecutor
from app.extract import extract_json_array

RUN_ID = 'colab-gemma4-01'   # unique per batch; kept on every row
MAX_WORKERS = 8
SHARD_SIZE = 96
OLLAMA_URL = 'http://localhost:11434/v1/chat/completions'

os.makedirs(RESULTS_DIR, exist_ok=True)
shard_base = len(done)
shard_path = os.path.join(RESULTS_DIR,
    f'reasonings_{shard_base:04d}-{shard_base + SHARD_SIZE - 1:04d}.json')
print('shard file:', shard_path)

def call_once(task):
    prompt = render_prompt(task['payload'], min_n=MIN_N, max_n=MAX_N,
                           generalize=True)
    r = requests.post(
        OLLAMA_URL,
        json={'model': MODEL,
              'messages': [{'role': 'user', 'content': prompt}],
              'temperature': 0.05,
              'stream': False},
        timeout=600,
    )
    r.raise_for_status()
    return r.json()['choices'][0]['message']['content'].strip()

def generate(task):
    last_err = None
    for attempt in range(2):   # light retry
        try:
            raw = call_once(task)
            status, _ = extract_json_array(raw)
            return {**task, 'run_id': RUN_ID, 'model': MODEL,
                    'raw_response': raw, 'parse_status': status}
        except Exception as e:
            last_err = e
            time.sleep(5 * (attempt + 1))
    return {**task, 'run_id': RUN_ID, 'model': MODEL, 'raw_response': '',
            'parse_status': 'error', 'error': str(last_err)[:200]}

results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = [pool.submit(generate, t) for t in tasks]
    n = 0
    for f in futures:
        res = f.result()
        results.append(res)
        n += 1
        with open(shard_path, 'w') as fh:      # incremental: survives resets
            json.dump(results, fh, indent=2, ensure_ascii=False)
        print(f"[{n}/{len(tasks)}] #{res['sentence_id']} {res['parse_status']} "
              f"({len(res['raw_response'])} chars)")

from collections import Counter
print('parse_status:', dict(Counter(r['parse_status'] for r in results)))
print('shard saved:', shard_path, f'({len(results)} rows)')


shard file: results/reasonings_0288-0383.json
[1/96] #15800 ok (281 chars)
[2/96] #23083 ok (399 chars)
[3/96] #35297 ok (317 chars)
[4/96] #50894 ok (482 chars)
[5/96] #47318 ok (454 chars)
[6/96] #33423 ok (387 chars)
[7/96] #21416 ok (425 chars)
[8/96] #12336 ok (447 chars)
[9/96] #35731 ok (576 chars)
[10/96] #38801 ok (479 chars)
[11/96] #43107 ok (338 chars)
[12/96] #50919 ok (503 chars)
[13/96] #43498 ok (442 chars)
[14/96] #51666 ok (487 chars)
[15/96] #52711 ok (475 chars)
[16/96] #13961 ok (579 chars)
[17/96] #51831 ok (430 chars)
[18/96] #24755 ok (358 chars)
[19/96] #54356 ok (501 chars)
[20/96] #32812 ok (605 chars)
[21/96] #18744 ok (504 chars)
[22/96] #24743 ok (323 chars)
[23/96] #25242 ok (472 chars)
[24/96] #35263 ok (430 chars)
[25/96] #22800 ok (457 chars)
[26/96] #17354 ok (462 chars)
[27/96] #48844 ok (314 chars)
[28/96] #19530 ok (468 chars)
[29/96] #37740 ok (464 chars)
[30/96] #51467 ok (399 chars)
[31/96] #38210 ok (460 chars)
[32/96] #55254 ok (431 chars)
[33

In [35]:
# 7. Download shard(s) + commit back (resume)
from google.colab import drive
drive.mount('/content/drive')

from google.colab import files

for f in sorted(os.listdir(RESULTS_DIR)):
    if f.startswith('reasonings_'):
        files.download(os.path.join(RESULTS_DIR, f))

# Commit the shard back to GitHub so the next Colab session resumes.
# Colab clones use HTTPS, so set a token first (once per session):
#   !git remote set-url origin https://<PAT>@github.com/jpeckenpaugh/qsbc.git
# then uncomment:
# !git add results/ && git commit -m 'add reasoning shard' && git push -q origin HEAD:main

# Optional: also stash a copy on Drive if mounted
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs('/content/drive/MyDrive/qsbc_results', exist_ok=True)
    for f in sorted(os.listdir(RESULTS_DIR)):
        if f.startswith('reasonings_'):
            !cp {os.path.join(RESULTS_DIR, f)} /content/drive/MyDrive/qsbc_results/


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>